# PoC V6.0 — Phases 1 & 2: Ground Truth + VQE with Diagnostics

This notebook implements the first two phases of the GNN-HVA v6.0 pipeline
using the new modular architecture:

- **Phase 1**: Classical ground truth via `HamiltonianBuilder` + `ClassicalSolver`
- **Phase 2**: VQE with diagnostic callbacks, expanded bounds [-π,π], descending sweep

System: 1D TFIM, N=6, HVA p=2, |+⟩^N initial state.

In [ ]:
import sys
from pathlib import Path

# Add src/ to sys.path so 'poc.v6' package is importable
_src = str(Path().resolve().parents[1])  # src/poc/v6 -> src/poc -> src
if _src not in sys.path:
    sys.path.insert(0, _src)

import numpy as np
import matplotlib.pyplot as plt
import logging

from poc.v6.hamiltonian_builder import HamiltonianBuilder, make_lattice
from poc.v6.classical_solver import ClassicalSolver
from poc.v6.hva_builder import HVACircuitBuilder
from poc.v6.vqe_optimizer import VQEOptimizer
from poc.v6.config import VQEConfig
from poc.v6.pipeline_utils import save_phase12_dataset

logging.basicConfig(level=logging.INFO)
np.random.seed(42)

# Configuration
N = 6
J = 1.0
p_layers = 2

# Non-uniform h-grid (V4 pattern)
h_coarse = np.arange(0.0, 0.8, 0.1)
h_dense = np.arange(0.8, 1.45, 0.05)
h_coarse2 = np.arange(1.5, 2.05, 0.1)
h_values = np.unique(np.concatenate([h_coarse, h_dense, h_coarse2]))

print(f'TFIM 1D: N={N}, J={J}, p={p_layers}, h ∈ [{h_values[0]}, {h_values[-1]}], {len(h_values)} points')

## Phase 1: Ground Truth Generation

In [ ]:
builder = HamiltonianBuilder()
solver = ClassicalSolver()

exact_data = []
print('Phase 1: Exact diagonalization...')
for h in h_values:
    lat_h = make_lattice('chain_1d', N, J=J, h=h)
    H = builder.build(lat_h)
    result = solver.solve(H, lat_h, method='exact')
    exact_data.append(result)

# Keep a base lattice reference for Phase 2 (topology/edges only)
base_lattice = make_lattice('chain_1d', N, J=J, h=1.0)

print(f'Phase 1 complete. {len(exact_data)} points solved.')
print(f'Gap range: [{min(r.gap for r in exact_data):.6f}, {max(r.gap for r in exact_data):.6f}]')

In [ ]:
# Phase 1 visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(h_values, [d.corr_zz for d in exact_data], 'o-', label=r'$\langle ZZ \rangle$')
ax1.plot(h_values, [d.mag_x for d in exact_data], 's-', label=r'$\langle X \rangle$')
ax1.axvline(x=1.0, color='red', linestyle='--', label='h_c (thermodynamic)')
ax1.set_xlabel('h/J'); ax1.set_ylabel('Observable')
ax1.set_title('Order Parameters'); ax1.legend(); ax1.grid(True)

ax2.plot(h_values, [d.gap for d in exact_data], 'D-', color='purple')
ax2.axvline(x=1.0, color='red', linestyle='--')
ax2.set_xlabel('h/J'); ax2.set_ylabel('Δ = E₁ - E₀')
ax2.set_title('Spectral Gap'); ax2.grid(True)

plt.tight_layout(); plt.show()

## Phase 2: VQE with Diagnostic Callbacks

In [ ]:
hva_builder = HVACircuitBuilder()
qc, theta = hva_builder.create(N, p_layers, base_lattice)

config = VQEConfig(
    p_layers=p_layers,
    n_restarts=3,
    maxiter=1000,
    ftol=1e-14,
    enable_callbacks=True,
)

optimizer = VQEOptimizer(config)

print(f'Phase 2: VQE descending sweep (p={p_layers}, bounds={config.bounds})...')
vqe_results = optimizer.descending_sweep(h_values, qc, base_lattice, exact_data)

# Summary
fids = [r.fidelity for r in vqe_results]
n_good = sum(1 for f in fids if f >= 0.995)
print(f'\nPhase 2 complete.')
print(f'Fidelity: avg={np.mean(fids)*100:.2f}%, min={np.min(fids)*100:.2f}%')
print(f'Points with fid ≥ 99.5%: {n_good}/{len(fids)}')

In [ ]:
# Phase 2 visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Fidelity
axes[0].plot(h_values, [r.fidelity * 100 for r in vqe_results], 'g^-')
axes[0].axhline(y=99.5, color='orange', linestyle=':', label='99.5% threshold')
axes[0].axvline(x=1.0, color='red', linestyle='--')
axes[0].set_xlabel('h/J'); axes[0].set_ylabel('Fidelity (%)')
axes[0].set_title('HVA Quality'); axes[0].legend(); axes[0].grid(True)

# Energy error
axes[1].semilogy(h_values, [r.energy_error for r in vqe_results], 'ro-')
axes[1].axvline(x=1.0, color='red', linestyle='--')
axes[1].set_xlabel('h/J'); axes[1].set_ylabel('|E_VQE - E_exact|')
axes[1].set_title('Energy Error'); axes[1].grid(True)

# θ_opt landscape
theta_opt = np.array([r.theta_opt for r in vqe_results])
for i in range(theta_opt.shape[1]):
    axes[2].plot(h_values, theta_opt[:, i], 'o-', label=f'θ_{i}')
axes[2].axvline(x=1.0, color='red', linestyle='--')
axes[2].set_xlabel('h/J'); axes[2].set_ylabel('θ_opt')
axes[2].set_title('Parameter Landscape'); axes[2].legend(); axes[2].grid(True)

plt.tight_layout(); plt.show()

## Save Dataset for Phase 3

In [ ]:
save_phase12_dataset(
    'phase1_phase2_tfim_N6_p2_v6.npz',
    h_values=h_values,
    J=J,
    n_qubits=N,
    p_layers=p_layers,
    ground_energies=np.array([d.ground_energy for d in exact_data]),
    gaps=np.array([d.gap for d in exact_data]),
    mag_x=np.array([d.mag_x for d in exact_data]),
    corr_zz=np.array([d.corr_zz for d in exact_data]),
    theta_opt=np.array([r.theta_opt for r in vqe_results]),
    vqe_energies=np.array([r.energy for r in vqe_results]),
    fidelities=np.array([r.fidelity for r in vqe_results]),
    per_site_mag_x=np.array([d.per_site_mag_x for d in exact_data]),
    per_bond_corr_zz=np.array([d.per_bond_corr_zz for d in exact_data]),
)
print('Dataset saved with v6.0 metadata.')